# 0824_peace_001_eda

Siemens AOI 데이터를 Polars로 분석하여 데이터 구조, 클래스 불균형, 검사 유형, 시간 변화, 특성 품질, 데이터 누수 가능성, 시간순 학습/평가 분할을 점검합니다.

- `class = 0`: AOI false call(오검출)
- `class = 1`: 수동검사에서 확인된 real defect(실제 불량)
- 모든 행은 AOI가 불량 의심으로 분류해 수동검사(MIS)로 보낸 사례이므로 별도의 AOI pass/fail 열은 없습니다.

### 핵심 결론 한눈에 보기

>현재 `dataset.csv`기준의 결론입니다. 이 데이터의 목표는 일반적인 정상/불량 탐지보다, **실제 불량을 놓치지 않으면서 AOI false call에 사용되는 수동검사량을 줄이는 것**입니다.

| 판단 영역 | 핵심 인사이트 | 분석·모델링 결정 |
|---|---|---|
| 데이터 품질 | 440,274행, Null/NaN 없음 | 기본 학습 품질은 양호 |
| 타겟 분포 | 실제 불량은 1.05% | Accuracy 대신 Recall·PR-AUC 사용 |
| 검사 유형 | 유형별 불량률과 유효 특성이 다름 | 통합 모델과 유형별 모델 비교 |
| 시간 변화 | Train 0.63% → Test 3.26% | 랜덤 분할 금지, 시간순 검증 |
| 특성 품질 | 전체 상수 특성 6개, 유형별 상수 특성 다수 | 매핑 적용 후 제거 |
| 운영 목표 | 실제 불량 미탐지 비용이 큼 | 최소 Recall을 먼저 정한 뒤 임계값 선택 |

## 0. 실행 환경

Polars가 설치되지 않은 환경에서는 다음 셀의 주석을 해제해 한 번만 설치하세요. 설치 후 커널을 재시작하면 됩니다.

In [1]:
# %pip install -q "polars>=1.0"
# %pip install --upgrade pip

In [2]:
from pathlib import Path
import json
import polars as pl

DATA_PATH = Path("../data/raw/dataset.csv")
MAPPING_PATH = Path("../data/raw/mapping.json")

assert DATA_PATH.exists(), f"파일을 찾을 수 없습니다: {DATA_PATH.resolve()}"
assert MAPPING_PATH.exists(), f"파일을 찾을 수 없습니다: {MAPPING_PATH.resolve()}"

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(12)
print(f"Polars {pl.__version__}")

Polars 1.43.2


## 1. LazyFrame으로 데이터 로딩

`scan_csv`는 필요한 열과 연산만 CSV 스캔 단계로 내려 보내므로 전체 파일을 즉시 메모리에 올리지 않습니다.

In [3]:
raw_lf = pl.scan_csv(DATA_PATH, try_parse_dates=False)
raw_columns = raw_lf.collect_schema().names()

# CSV 첫 번째 열은 저장된 인덱스이며 열 이름이 비어 있다.
if raw_columns[0] == "":
    raw_lf = raw_lf.rename({"": "row_id"})
elif raw_columns[0].lower().startswith("unnamed"):
    raw_lf = raw_lf.rename({raw_columns[0]: "row_id"})

lf = raw_lf.with_columns(
    pl.col("timestamp").str.to_datetime(time_zone="UTC", strict=False).alias("timestamp"),
    pl.col("class").cast(pl.Int8),
    pl.col("inspection_type").cast(pl.Int8),
)

schema = lf.collect_schema()
print(f"열 수: {len(schema)}")
schema

열 수: 78


Schema([('row_id', Int64),
        ('timestamp', Datetime(time_unit='us', time_zone='UTC')),
        ('class', Int8),
        ('inspection_type', Int8),
        ('meta_feat1', Int64),
        ('meta_feat2', Int64),
        ('meta_feat3', Int64),
        ('meta_feat4', Int64),
        ('inspection_feat1', Float64),
        ('inspection_feat2', Float64),
        ('inspection_feat3', Float64),
        ('inspection_feat4', Float64),
        ('inspection_feat5', Float64),
        ('inspection_feat6', Float64),
        ('inspection_feat7', Float64),
        ('inspection_feat8', Float64),
        ('inspection_feat9', Float64),
        ('inspection_feat10', Float64),
        ('inspection_feat11', Float64),
        ('inspection_feat12', Float64),
        ('inspection_feat13', Float64),
        ('inspection_feat14', Float64),
        ('inspection_feat15', Float64),
        ('inspection_feat16', Float64),
        ('inspection_feat17', Float64),
        ('inspection_feat18', Float64),
        ('in

## 2. 기본 품질 점검

행 수, 열 수, 기간, 인덱스 고유성, 열별 결측값을 확인합니다. Polars는 `NaN`과 `null`을 구분하므로 수치형 열의 NaN도 별도로 검사합니다.

In [4]:
overview = lf.select(
    pl.len().alias("rows"),
    pl.col("row_id").n_unique().alias("unique_row_ids"),
    pl.col("timestamp").min().alias("start_time"),
    pl.col("timestamp").max().alias("end_time"),
).collect()
overview.with_columns(pl.lit(len(schema)).alias("columns"))

rows,unique_row_ids,start_time,end_time,columns
u32,u32,"datetime[μs, UTC]","datetime[μs, UTC]",i32
440274,440274,1970-06-23 03:58:55 UTC,1970-11-02 14:21:28 UTC,78


In [5]:
null_counts = (
    lf.select(pl.all().null_count())
    .collect()
    .transpose(include_header=True, header_name="column", column_names=["null_count"])
    .filter(pl.col("null_count") > 0)
    .sort("null_count", descending=True)
)

numeric_columns = [name for name, dtype in schema.items() if dtype.is_numeric()]
nan_counts = (
    lf.select([pl.col(c).is_nan().sum().alias(c) for c in numeric_columns])
    .collect()
    .transpose(include_header=True, header_name="column", column_names=["nan_count"])
    .filter(pl.col("nan_count") > 0)
    .sort("nan_count", descending=True)
)
print("null이 있는 열")
display(null_counts)
print("NaN이 있는 수치형 열")
display(nan_counts)

null이 있는 열


column,null_count
str,u32


NaN이 있는 수치형 열


column,nan_count
str,u32


## 3. 목표 변수와 클래스 불균형

단순 정확도는 거의 모든 행을 `class=0`으로 예측해도 높게 나오므로 핵심 성능 지표로 사용하면 안 됩니다.

In [6]:
class_summary = (
    lf.group_by("class")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum()).alias("ratio"),
        pl.when(pl.col("class") == 0)
        .then(pl.lit("false_call"))
        .otherwise(pl.lit("real_defect"))
        .alias("meaning"),
    )
    .sort("class")
    .collect()
)
class_summary.with_columns((pl.col("ratio") * 100).round(3).alias("percent"))

class,count,ratio,meaning,percent
i8,u32,f64,str,f64
0,435652,0.989502,"""false_call""",98.95
1,4622,0.010498,"""real_defect""",1.05


### 해석 — 정확도가 높아 보여도 좋은 모델이 아니다

- 실제 불량은 4,622건(1.05%)으로, false call과 약 94:1의 불균형이다.
- 모든 행을 false call로 예측해도 Accuracy가 98.95%이므로 Accuracy는 핵심 성능지표로 사용하지 않는다.
- 모델은 실제 불량 Recall을 안전 제약조건으로 두고, 그 조건을 만족하는 범위에서 Precision과 false-call reduction을 최적화해야 한다.

## 4. 검사 유형별 데이터 수와 실제 불량률

`inspection_type`은 출력값이 아니라 어떤 검사 방식과 특성 집합이 적용됐는지를 나타내는 입력 변수입니다.

In [7]:
type_summary = (
    lf.group_by("inspection_type")
    .agg(
        pl.len().alias("rows"),
        pl.col("class").sum().alias("real_defects"),
        pl.col("class").mean().alias("defect_rate"),
    )
    .with_columns((pl.col("defect_rate") * 100).round(3).alias("defect_percent"))
    .sort("inspection_type")
    .collect()
)
type_summary

inspection_type,rows,real_defects,defect_rate,defect_percent
i8,u32,i64,f64,f64
0,97053,318,0.003277,0.328
1,57673,1578,0.027361,2.736
2,128174,1346,0.010501,1.05
3,151931,1281,0.008431,0.843
4,5443,99,0.018188,1.819


### 해석 — `inspection_type`은 핵심 분할 축이다

- 유형 1의 불량률은 2.736%, 유형 0은 0.328%로 약 8.3배 차이가 난다.
- 단일 임계값을 모든 검사 유형에 공통 적용하면 유형별 경보 품질이 크게 달라질 수 있다.
- `inspection_type`을 범주형 입력으로 포함한 통합 모델과 유형별 개별 모델을 모두 시간순 검증으로 비교한다.

## 5. 시간에 따른 분포 변화

일별 데이터량과 불량률을 계산합니다. 급격한 변화는 공정 변화, 제품 믹스 변화 또는 라벨링 변화일 수 있습니다.

In [8]:
daily_summary = (
    lf.with_columns(pl.col("timestamp").dt.date().alias("date"))
    .group_by("date")
    .agg(
        pl.len().alias("rows"),
        pl.col("class").sum().alias("real_defects"),
        pl.col("class").mean().alias("defect_rate"),
        pl.col("inspection_type").n_unique().alias("inspection_types"),
    )
    .sort("date")
    .collect()
)
daily_summary

date,rows,real_defects,defect_rate,inspection_types
date,u32,i64,f64,u32
1970-06-23,3131,6,0.001916,4
1970-06-24,4900,6,0.001224,5
1970-06-26,893,0,0.0,5
1970-06-27,1178,0,0.0,5
1970-06-28,4714,5,0.001061,5
1970-06-29,2289,1,0.000437,5
1970-07-01,2250,5,0.002222,5
1970-07-03,149,1,0.006711,5
1970-07-04,3969,35,0.008818,5


In [9]:
# 불량률 변화가 가장 큰 날짜를 우선 점검한다.
daily_shift = (
    daily_summary.with_columns(
        pl.col("defect_rate").diff().alias("rate_change")
    )
    .with_columns(pl.col("rate_change").abs().alias("abs_rate_change"))
    .filter(pl.col("abs_rate_change").is_not_null())
    .sort("abs_rate_change", descending=True)
    .head(15)
)
daily_shift

date,rows,real_defects,defect_rate,inspection_types,rate_change,abs_rate_change
date,u32,i64,f64,u32,f64,f64
1970-08-15,463,68,0.146868,4,0.146868,0.146868
1970-08-16,3585,5,0.001395,5,-0.145474,0.145474
1970-10-27,5040,637,0.126389,5,0.121703,0.121703
1970-10-28,13052,641,0.049111,5,-0.077278,0.077278
1970-07-25,3779,251,0.06642,5,0.06642,0.06642
1970-07-27,757,0,0.0,5,-0.06642,0.06642
1970-10-17,8295,47,0.005666,5,-0.059149,0.059149
1970-09-05,1708,98,0.057377,4,0.057377,0.057377
1970-10-20,11680,696,0.059589,5,0.055776,0.055776


### 해석 — 시간 드리프트가 가장 큰 모델링 위험이다

- 일별 불량률은 0%에서 14%대까지 급격히 변하며, 8월 15일과 10월 27일 전후에 특히 큰 변화가 보인다.
- 이는 공정 상태, 제품 믹스, 검사 조건 또는 라벨링 기준이 바뀌었을 가능성을 의미한다.
- 랜덤 분할은 미래 공정 상태를 학습 데이터에 섞어 실제 운영 성능을 과대평가할 수 있으므로 사용하지 않는다.

## 6. `mapping.json`과 검사 유형별 유효 특성

검사 유형마다 의미 있는 `inspection_feat*`가 다르므로 모든 특성을 무조건 함께 사용하는 대신 매핑에 맞춰 분석합니다.

In [10]:
with MAPPING_PATH.open(encoding="utf-8") as f:
    feature_mapping = json.load(f)

mapping_summary = pl.DataFrame(
    {
        "inspection_type": [int(k) for k in feature_mapping],
        "mapped_feature_count": [len(v) for v in feature_mapping.values()],
    }
).sort("inspection_type")

mapped_union = sorted(set().union(*(set(v) for v in feature_mapping.values())))
mapped_intersection = sorted(set.intersection(*(set(v) for v in feature_mapping.values())))
missing_mapped_columns = sorted(set(mapped_union) - set(schema.names()))

display(mapping_summary)
print(f"매핑에 등장하는 고유 특성: {len(mapped_union)}개")
print(f"모든 검사 유형의 공통 특성: {len(mapped_intersection)}개")
print(f"CSV에 없는 매핑 특성: {missing_mapped_columns}")

inspection_type,mapped_feature_count
i64,i64
0,44
1,52
2,65
3,65
4,21


매핑에 등장하는 고유 특성: 65개
모든 검사 유형의 공통 특성: 17개
CSV에 없는 매핑 특성: []


### 해석 — 특성 유효성을 검사 유형별로 판단해야 한다

- 유형별 유효 특성은 21~65개로 다르고, 모든 유형에 공통인 특성은 17개뿐이다.
- 다른 검사 유형에서 사용되지 않아 0으로 채워진 값을 실제 측정값 0으로 해석하면 안 된다.
- `mapping.json`으로 유효 특성을 선택하거나, 통합 모델에서 특성 유효 마스크를 함께 사용하는 방법을 검토한다.

## 7. 검사 유형별 특성과 `class`의 관계

아래 함수는 해당 검사 유형에 유효한 특성만 대상으로 평균, 표준편차, 고유값 수, `class`와의 Pearson 상관계수를 계산합니다. 상수·극저분산 특성은 제외합니다. 상관계수는 탐색 지표일 뿐 인과관계나 실제 예측력을 보장하지 않습니다.

In [11]:
def inspect_features(inspection_type: int, top_n: int = 20) -> pl.DataFrame:
    features = feature_mapping[str(inspection_type)]
    subset = lf.filter(pl.col("inspection_type") == inspection_type)

    correlations = (
        subset.select([pl.corr(c, "class").alias(c) for c in features])
        .collect()
        .transpose(include_header=True, header_name="feature", column_names=["class_corr"])
    )
    feature_stats = (
        subset.select(
            [pl.col(c).mean().alias(f"{c}__mean") for c in features]
            + [pl.col(c).std().alias(f"{c}__std") for c in features]
            + [pl.col(c).n_unique().alias(f"{c}__n_unique") for c in features]
        )
        .collect()
    )
    rows = []
    for feature in features:
        rows.append(
            {
                "feature": feature,
                "mean": feature_stats.item(0, f"{feature}__mean"),
                "std": feature_stats.item(0, f"{feature}__std"),
                "n_unique": feature_stats.item(0, f"{feature}__n_unique"),
            }
        )
    return (
        pl.DataFrame(rows)
        .join(correlations, on="feature", how="left")
        .filter(
            (pl.col("n_unique") > 1)
            & (pl.col("std").abs() > 1e-12)
            & pl.col("class_corr").is_finite()
        )
        .with_columns(pl.col("class_corr").abs().alias("abs_class_corr"))
        .sort("abs_class_corr", descending=True, nulls_last=True)
        .head(top_n)
    )

inspect_features(inspection_type=0, top_n=20)

feature,mean,std,n_unique,class_corr,abs_class_corr
str,f64,f64,i64,f64,f64
"""inspection_feat49""",0.90939,0.193855,3,-0.052723,0.052723
"""inspection_feat12""",0.000871,0.00037,2,-0.044349,0.044349
"""inspection_feat4""",0.429995,0.022458,189,-0.029014,0.029014
"""inspection_feat23""",0.302381,0.241382,222,-0.022837,0.022837
"""inspection_feat42""",0.218473,0.213762,230,-0.017142,0.017142
"""inspection_feat41""",0.199147,0.190047,226,-0.015606,0.015606
"""inspection_feat40""",0.0411,0.09925,180,0.015225,0.015225
"""inspection_feat27""",0.070102,0.160843,225,-0.013152,0.013152
"""inspection_feat54""",0.032124,0.069094,112,-0.012535,0.012535


다른 검사 유형도 같은 함수로 확인합니다. 한 모델에 모든 유형을 섞기 전에 유형별 모델과 통합 모델을 모두 비교하는 것이 좋습니다.

In [12]:
for inspection_type in range(5):
    print(f"inspection_type={inspection_type}")
    display(inspect_features(inspection_type, top_n=10))

inspection_type=0


feature,mean,std,n_unique,class_corr,abs_class_corr
str,f64,f64,i64,f64,f64
"""inspection_feat49""",0.90939,0.193855,3,-0.052723,0.052723
"""inspection_feat12""",0.000871,0.00037,2,-0.044349,0.044349
"""inspection_feat4""",0.429995,0.022458,189,-0.029014,0.029014
"""inspection_feat23""",0.302381,0.241382,222,-0.022837,0.022837
"""inspection_feat42""",0.218473,0.213762,230,-0.017142,0.017142
"""inspection_feat41""",0.199147,0.190047,226,-0.015606,0.015606
"""inspection_feat40""",0.0411,0.09925,180,0.015225,0.015225
"""inspection_feat27""",0.070102,0.160843,225,-0.013152,0.013152
"""inspection_feat54""",0.032124,0.069094,112,-0.012535,0.012535


inspection_type=1


feature,mean,std,n_unique,class_corr,abs_class_corr
str,f64,f64,i64,f64,f64
"""inspection_feat48""",0.070899,0.184846,241,0.288185,0.288185
"""inspection_feat24""",0.122948,0.186108,241,0.182647,0.182647
"""inspection_feat88""",0.000936,0.023038,101,0.147768,0.147768
"""inspection_feat97""",0.001091,0.02506,108,0.144615,0.144615
"""inspection_feat90""",0.000908,0.023793,88,0.125046,0.125046
"""inspection_feat98""",0.001167,0.026542,100,0.122884,0.122884
"""inspection_feat89""",0.001064,0.02494,89,0.121647,0.121647
"""inspection_feat99""",0.001183,0.028425,98,0.115606,0.115606
"""inspection_feat7""",0.971762,0.016274,96,-0.067199,0.067199


inspection_type=2


feature,mean,std,n_unique,class_corr,abs_class_corr
str,f64,f64,i64,f64,f64
"""inspection_feat28""",0.409525,0.153686,2114,-0.047081,0.047081
"""inspection_feat95""",0.355301,0.198113,41,-0.041494,0.041494
"""inspection_feat12""",0.062537,0.119542,303,-0.02403,0.02403
"""inspection_feat8""",0.36353,0.0474,101,-0.023257,0.023257
"""inspection_feat20""",0.477922,0.001739,3,-0.020303,0.020303
"""inspection_feat9""",0.634228,0.14983,101,0.018256,0.018256
"""inspection_feat3""",0.484081,0.052459,153,-0.01679,0.01679
"""inspection_feat93""",0.175363,0.12676,174,-0.016722,0.016722
"""inspection_feat18""",0.089138,0.145899,104,-0.01663,0.01663


inspection_type=3


feature,mean,std,n_unique,class_corr,abs_class_corr
str,f64,f64,i64,f64,f64
"""inspection_feat95""",0.386361,0.236693,39,-0.062888,0.062888
"""inspection_feat12""",0.081392,0.106858,182,-0.047313,0.047313
"""inspection_feat22""",0.866072,0.080149,135,-0.042033,0.042033
"""inspection_feat96""",0.237751,0.1004,40,-0.041598,0.041598
"""inspection_feat28""",0.260422,0.059875,767,-0.036078,0.036078
"""inspection_feat93""",0.242741,0.197998,222,-0.033333,0.033333
"""inspection_feat17""",0.096201,0.092671,410,-0.03245,0.03245
"""inspection_feat94""",0.164729,0.130488,597,-0.032283,0.032283
"""inspection_feat4""",0.433536,0.027394,553,0.026756,0.026756


inspection_type=4


feature,mean,std,n_unique,class_corr,abs_class_corr
str,f64,f64,i64,f64,f64
"""inspection_feat7""",0.967825,0.004762,9,-0.064314,0.064314
"""inspection_feat1""",0.593736,0.013639,162,0.027502,0.027502
"""inspection_feat9""",0.563716,0.004094,10,0.023622,0.023622
"""inspection_feat3""",0.570486,0.023922,211,0.020666,0.020666
"""inspection_feat25""",0.009942,0.067981,48,-0.019907,0.019907
"""inspection_feat2""",0.783747,0.132514,176,0.015317,0.015317
"""inspection_feat4""",0.43177,0.004017,90,0.014972,0.014972
"""inspection_feat5""",0.514054,0.074853,293,-0.013359,0.013359
"""inspection_feat32""",0.514054,0.074853,293,-0.013359,0.013359


### 해석 — 단일 상관계수만으로 예측력을 판단하지 않는다

- 유형 1에서 `inspection_feat48`(상관계수 0.288), `inspection_feat24`(0.183)가 가장 뚜렷한 선형 신호를 보였고, 희소 특성 88·99가 그 뒤를 이었다.
- 반면 유형 0·2·3·4의 최상위 절대 상관계수는 약 0.05~0.06 수준으로, 단일 특성만으로는 불량을 잘 나누기 어렵다.
- 검사 유형 내에서 대부분의 단일 특성 상관계수는 크지 않으며, 이는 비선형 관계와 특성 조합이 중요할 수 있음을 의미한다.
- 상수·극저분산 특성의 상관계수는 부동소수점 오차로 그럴듯한 값이 나올 수 있어, 해당 특성을 먼저 제거한 후 순위를 계산한다.
- 최종 특성 판단은 시간순 Validation의 permutation importance 또는 SHAP과 같은 모델 기반 분석으로 보완한다.

## 8. 상수·준상수 특성 및 누수 후보 점검

전체 데이터에서 값이 하나뿐인 열은 모델에 도움이 되지 않습니다. 또한 값 하나가 거의 특정 클래스와 일치하거나 시간·메타데이터가 정답을 지나치게 잘 분리하면 데이터 누수를 의심해야 합니다.

In [13]:
candidate_features = [c for c in schema.names() if c.startswith(("meta_feat", "inspection_feat"))]
unique_counts = (
    lf.select([pl.col(c).n_unique().alias(c) for c in candidate_features])
    .collect()
    .transpose(include_header=True, header_name="feature", column_names=["n_unique"])
    .sort("n_unique")
)
constant_features = unique_counts.filter(pl.col("n_unique") <= 1)
display(constant_features)
display(unique_counts.head(20))

feature,n_unique
str,u32
"""inspection_feat13""",1
"""inspection_feat16""",1
"""inspection_feat21""",1
"""inspection_feat71""",1
"""inspection_feat72""",1
"""inspection_feat78""",1


feature,n_unique
str,u32
"""inspection_feat13""",1
"""inspection_feat16""",1
"""inspection_feat21""",1
"""inspection_feat71""",1
"""inspection_feat72""",1
"""inspection_feat78""",1
"""meta_feat3""",2
"""inspection_feat30""",3
"""inspection_feat31""",3


In [14]:
# 익명화 메타 특성의 값별 표본 수와 불량률을 확인한다.
meta_columns = [c for c in schema.names() if c.startswith("meta_feat")]
for column in meta_columns:
    print(column)
    display(
        lf.group_by(column)
        .agg(
            pl.len().alias("rows"),
            pl.col("class").sum().alias("real_defects"),
            pl.col("class").mean().alias("defect_rate"),
        )
        .sort("rows", descending=True)
        .collect()
        .head(20)
    )

meta_feat1


meta_feat1,rows,real_defects,defect_rate
i64,u32,i64,f64
1,60770,124,0.00204
2,49579,290,0.005849
12,38380,66,0.00172
8,31538,128,0.004059
52,27291,20,0.000733
16,20952,292,0.013937
13,14712,78,0.005302
18,13461,61,0.004532
31,12992,3,0.000231


meta_feat2


meta_feat2,rows,real_defects,defect_rate
i64,u32,i64,f64
3,257529,2953,0.011467
1,126274,923,0.00731
2,55303,745,0.013471
5,940,0,0.0
4,228,1,0.004386


meta_feat3


meta_feat3,rows,real_defects,defect_rate
i64,u32,i64,f64
1,282801,2686,0.009498
0,157473,1936,0.012294


meta_feat4


meta_feat4,rows,real_defects,defect_rate
i64,u32,i64,f64
2,51903,386,0.007437
3,37788,136,0.003599
16,30997,160,0.005162
28,28723,700,0.024371
1,26993,749,0.027748
12,26457,17,0.000643
26,26331,13,0.000494
7,25436,161,0.00633
23,23598,60,0.002543


### 해석 — 메타 특성은 유용할 수 있지만 누수 검증이 먼저다

- 전체 상수 특성 6개는 정보량이 없으므로 제거한다. 검사 유형별로 상수가 되는 특성도 해당 유형 모델에서 제거한다.
- `meta_feat1=3`은 9.46%, `meta_feat1=27`은 5.02%등 특정 메타 값에서 전체 1.05%보다 훨씬 높은 불량률이 관찰된다.
- 이 패턴은 제품/공정 군집 신호일 수도 있지만, 수동검사 후에 생성되는 정보나 배치 ID의 대리 변수라면 누수가 된다. 익명화 이전 의미와 예측 시점의 사용 가능 여부를 확인한다.

## 9. 시간순 학습·검증·테스트 분할

무작위 분할은 미래 공정 상태가 학습 데이터에 섞이는 누수를 만들 수 있습니다. 시간순 70%/15%/15% 분할로 과거에서 학습하고 미래에서 평가합니다. 동일 timestamp가 서로 다른 분할에 걸치지 않도록 시간 경계로 나눕니다.

In [15]:
split_points = lf.select(
    pl.col("timestamp").quantile(0.70).alias("train_end"),
    pl.col("timestamp").quantile(0.85).alias("valid_end"),
).collect()
train_end = split_points.item(0, "train_end")
valid_end = split_points.item(0, "valid_end")

train_lf = lf.filter(pl.col("timestamp") <= train_end)
valid_lf = lf.filter((pl.col("timestamp") > train_end) & (pl.col("timestamp") <= valid_end))
test_lf = lf.filter(pl.col("timestamp") > valid_end)

def split_summary(name: str, frame: pl.LazyFrame) -> pl.DataFrame:
    return frame.select(
        pl.lit(name).alias("split"),
        pl.len().alias("rows"),
        pl.col("timestamp").min().alias("start_time"),
        pl.col("timestamp").max().alias("end_time"),
        pl.col("class").sum().alias("real_defects"),
        pl.col("class").mean().alias("defect_rate"),
    ).collect()

pl.concat([
    split_summary("train", train_lf),
    split_summary("validation", valid_lf),
    split_summary("test", test_lf),
])

split,rows,start_time,end_time,real_defects,defect_rate
str,u32,"datetime[μs, UTC]","datetime[μs, UTC]",i64,f64
"""train""",308196,1970-06-23 03:58:55 UTC,1970-10-05 00:29:59 UTC,1940,0.006295
"""validation""",66041,1970-10-05 00:30:30 UTC,1970-10-19 07:29:16 UTC,532,0.008056
"""test""",66037,1970-10-19 07:29:55 UTC,1970-11-02 14:21:28 UTC,2150,0.032558


### 해석 — 테스트 구간은 학습 구간과 다른 운영 환경이다

| 구간 | 불량률 | Train 대비 |
|---|---:|---:|
| Train | 0.630% | 1.0배 |
| Validation | 0.806% | 1.3배 |
| Test | 3.256% | 5.2배 |

Test는 전체 행의 약 15%이지만 전체 실제 불량의 약 46.5%(2,150/4,622)를 포함한다. 따라서 한 번의 성능 숫자만 보지 말고, 일/주 단위 성능과 검사 유형별 성능 변화를 함께 모니터링해야 한다.

## 10. 평가 기준

모델링 단계에서는 다음 순서로 판단하는 것이 안전합니다.

1. **실제 불량 recall (`class=1`)**: 실제 불량을 false call로 잘못 넘기지 않는지 확인
2. **PR-AUC**: 극심한 클래스 불균형에서 전반적인 순위 품질 평가
3. **Precision 및 F1**: 불량 예측의 신뢰도와 recall 간 균형 확인
4. **False-call reduction rate**: 안전 기준을 만족하면서 줄일 수 있는 수동검사량 측정

운영 임계값은 정확도 최대화가 아니라 `class=1` recall의 최소 허용치를 먼저 정한 후 선택해야 합니다. 검사 유형별 표본 수와 불량률이 다르므로 전체 지표뿐 아니라 `inspection_type`별 지표도 반드시 보고해야 합니다.

## 11. 분석 체크리스트

- 클래스 불균형과 검사 유형별 불량률 차이를 확인했는가?
- `mapping.json`에 따라 각 검사 유형의 유효 특성만 사용했는가?
- 시간에 따른 데이터량·불량률·특성 분포 변화를 확인했는가?
- 상수 특성, 지나치게 강한 대리 변수, 미래 정보 누수를 제거했는가?
- 무작위 분할이 아닌 시간순 분할로 미래 구간을 평가했는가?
- 전체 지표와 검사 유형별 recall, precision, PR-AUC를 함께 확인했는가?

## 12. 종합 인사이트와 다음 단계

### 종합 판단

이 데이터로 **실제 불량 탐지 및 false-call 절감 모델을 개발할 수 있다.** 다만 업무적 성공은 단순 분류 정확도가 아니라, 미탐지를 안전 수준 이하로 유지하면서 얼마나 많은 수동검사를 줄이는지로 판단해야 한다.

| 우선순위 | 다음 단계 | 이유 |
|---:|---|---|
| 1 | 메타 특성의 원래 의미와 예측 시점 사용 가능성 확인 | 데이터 누수 방지 |
| 2 | 상수 특성 제거 및 `mapping.json` 기반 유효 특성 구성 | 검사 유형별 0 값 오해 방지 |
| 3 | 통합 모델과 검사 유형별 모델 학습 | 유형별 불량률·특성 차이 반영 |
| 4 | 시간순 Validation에서 임계값 선택 | 운영 Recall 제약조건 반영 |
| 5 | Test에서 전체·유형별·일자별 성능 보고 | 후반 드리프에서의 안정성 확인 |
| 6 | Recall 기준별 false-call reduction 곡선 생성 | 안전성과 절감 효과를 운영 언어로 연결 |

### 최종 보고 형식

모델 결과는 `Recall ≥ 목표치`를 만족하는 임계값에서 **PR-AUC, Precision, 미탐지 건수, false-call reduction rate, 일일 수동검사 절감량**을 함께 보고한다.